# Core Strategy Geometry Audit (аудит геометрии базовых стратегий)

Цель: искать не только прибыльные `core strategy (базовая стратегия)`, а стратегии с эксплуатируемой формой `PnL (прибыль и убыток)`: устойчивый рост, устойчивое падение, возвратность, сезонность, серийность или анти-связь между секторами.

Ноутбук использует уже созданные результаты `priority_groups_stat_arb_audit` и общий `backtest_protocol.yaml`.

In [55]:
from pathlib import Path
import json
import math
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.ioff()

try:
    from numba import njit
    HAS_NUMBA = True
except Exception as exc:
    HAS_NUMBA = False
    print('numba недоступна, fallback на обычный Python/NumPy:', repr(exc))
    def njit(*args, **kwargs):
        if args and callable(args[0]):
            return args[0]
        def wrapper(func):
            return func
        return wrapper

BASE_DIR = Path.cwd()
PRIORITY_DIR = BASE_DIR / 'priority_groups_stat_arb_audit'
PROTOCOL_PATH = BASE_DIR / 'backtest_protocol.yaml'
PANEL_PATH = PRIORITY_DIR / '04_active_close_panel_30min.parquet'
SPECS_PATH = PRIORITY_DIR / '01_contract_specs_and_multipliers.csv'
GROUP_MAP_PATH = PRIORITY_DIR / '02_priority_group_map.csv'
PAIR_STATS_PATH = PRIORITY_DIR / '06_pair_statistics.csv'

OUTPUT_DIR = BASE_DIR / 'core_strategy_geometry_audit'
FIG_DIR = OUTPUT_DIR / 'figures'
PATH_DIR = OUTPUT_DIR / 'paths'
OUTPUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)
PATH_DIR.mkdir(exist_ok=True)

MAX_PLOT_POINTS = 6000
SAVE_DOWNSAMPLED_PNL_PATHS = True

print('HAS_NUMBA =', HAS_NUMBA)
OUTPUT_DIR

HAS_NUMBA = True


WindowsPath('C:/Users/jack/Jupyter_files/QUANT_TRAIDING/Fiter_2options/core_strategy_geometry_audit')

In [56]:
def read_protocol_config(path):
    try:
        import yaml
        with path.open('r', encoding='utf-8') as f:
            return yaml.safe_load(f)
    except Exception:
        cfg = {'splits': {}, 'cost_model': {}, 'selection_policy': {}}
        section = None
        split_name = None
        for raw in path.read_text(encoding='utf-8').splitlines():
            line = raw.split('#', 1)[0].rstrip()
            if not line.strip():
                continue
            if not raw.startswith(' '):
                key = line.rstrip(':')
                section = key if key in cfg else None
                if ':' in line and section is None:
                    k, v = line.split(':', 1)
                    cfg[k.strip()] = v.strip().strip('"')
                continue
            stripped = line.strip()
            if section == 'splits':
                if stripped.endswith(':'):
                    split_name = stripped[:-1]
                    cfg['splits'][split_name] = {}
                elif split_name and ':' in stripped:
                    k, v = stripped.split(':', 1)
                    cfg['splits'][split_name][k.strip()] = v.strip().strip('"')
            elif section in ['cost_model', 'selection_policy'] and ':' in stripped:
                k, v = stripped.split(':', 1)
                v = v.strip().strip('"')
                try:
                    v = float(v)
                except ValueError:
                    pass
                cfg[section][k.strip()] = v
        return cfg

protocol_cfg = read_protocol_config(PROTOCOL_PATH)
splits = {k: {'start': pd.Timestamp(v['start']), 'end': pd.Timestamp(v['end'])} for k, v in protocol_cfg['splits'].items()}
COST_PER_UNIT_TURNOVER = float(protocol_cfg.get('cost_model', {}).get('cost_per_unit_turnover', 0.00005))

def filter_by_split(obj, split_name):
    spec = splits[split_name]
    return obj.loc[(obj.index >= spec['start']) & (obj.index <= spec['end'])]

split_table = pd.DataFrame([{'split': k, **v} for k, v in splits.items()])
split_table.to_csv(OUTPUT_DIR / '00_protocol_splits.csv', index=False)
split_table

,split,start,end
0,train,2008-05-01,2016-12-31 23:59:59
1,calibration,2017-01-01,2019-12-31 23:59:59
2,test,2020-01-01,2023-08-10 23:59:59


In [57]:
panel_30m = pd.read_parquet(PANEL_PATH).sort_index()
specs = pd.read_csv(SPECS_PATH)
group_map = pd.read_csv(GROUP_MAP_PATH)
pair_stats = pd.read_csv(PAIR_STATS_PATH)

market_to_group = group_map.drop_duplicates('market').set_index('market')['group'].to_dict()
market_to_sector = specs.set_index('market')['sector'].to_dict()

valid_cols = [c for c in panel_30m.columns if c in market_to_group]
panel_30m = panel_30m[valid_cols]
returns_30m = np.log(panel_30m).diff()
daily_prices = panel_30m.resample('1D').last().dropna(how='all')
daily_returns = np.log(daily_prices).diff()

pd.DataFrame({
    'market': panel_30m.columns,
    'group': [market_to_group.get(c) for c in panel_30m.columns],
    'sector': [market_to_sector.get(c) for c in panel_30m.columns],
    'non_missing_share': panel_30m.notna().mean().values,
}).to_csv(OUTPUT_DIR / '01_loaded_panel_meta.csv', index=False)

panel_30m.shape, daily_prices.shape

((180979, 24), (4753, 24))

## Sector and Cross-Sector Correlation (секторная и межсекторная корреляция)

Здесь мы намеренно смотрим шире, чем классические товарные пары. Отрицательная `correlation (корреляция)` и сильный общий секторный тренд могут дать core-стратегии с отрицательным или режимным `PnL (прибыль и убыток)`, которые потом интересны для `put (пут)`-слоя.

In [58]:
group_returns = {}
for group, cols in pd.Series(market_to_group).groupby(pd.Series(market_to_group)).groups.items():
    markets = [m for m in cols if m in daily_returns.columns]
    if len(markets) >= 2:
        group_returns[group] = daily_returns[markets].mean(axis=1)
group_returns = pd.DataFrame(group_returns).dropna(how='all')
sector_corr = group_returns.corr()
sector_corr.to_csv(OUTPUT_DIR / '02_sector_return_correlation.csv')

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(sector_corr, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(sector_corr.columns)), sector_corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(sector_corr.index)), sector_corr.index)
ax.set_title('Sector return correlation (корреляция доходностей секторов)')
fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
fig.savefig(FIG_DIR / 'sector_return_correlation.png', dpi=150, bbox_inches='tight')
plt.close(fig)

market_corr = daily_returns.corr()
pairs = []
cols = list(market_corr.columns)
for i, a in enumerate(cols):
    for b in cols[i+1:]:
        corr = market_corr.loc[a, b]
        pairs.append({
            'a': a, 'b': b,
            'group_a': market_to_group.get(a), 'group_b': market_to_group.get(b),
            'sector_a': market_to_sector.get(a), 'sector_b': market_to_sector.get(b),
            'daily_return_corr': corr,
            'abs_corr': abs(corr),
            'is_cross_group': market_to_group.get(a) != market_to_group.get(b),
            'is_negative_corr': corr < 0,
        })
cross_corr_pairs = pd.DataFrame(pairs).sort_values('daily_return_corr')
cross_corr_pairs.to_csv(OUTPUT_DIR / '03_all_market_daily_correlation_pairs.csv', index=False)
negative_cross_pairs = cross_corr_pairs.query("is_cross_group and daily_return_corr < -0.20").copy()
negative_cross_pairs.to_csv(OUTPUT_DIR / '04_negative_cross_sector_pairs.csv', index=False)
negative_cross_pairs.head(25)

,a,b,group_a,group_b,sector_a,sector_b,daily_return_corr,abs_corr,is_cross_group,is_negative_corr
47,CC,DX,softs_low_priority,fx_usd_factor,Softs,FX,-0.205082,0.205082,True,True


In [59]:
@njit
def rolling_mean_std(x, window):
    n = len(x)
    mean = np.empty(n, dtype=np.float64)
    std = np.empty(n, dtype=np.float64)
    for i in range(n):
        mean[i] = np.nan
        std[i] = np.nan
    for i in range(window - 1, n):
        s = 0.0
        ss = 0.0
        c = 0
        for j in range(i - window + 1, i + 1):
            if np.isfinite(x[j]):
                s += x[j]
                ss += x[j] * x[j]
                c += 1
        if c >= max(10, window // 3):
            m = s / c
            v = max(ss / c - m * m, 0.0)
            mean[i] = m
            std[i] = math.sqrt(v)
    return mean, std

@njit
def strategy_from_signal(asset_ret, signal, cost_per_turnover):
    n = len(asset_ret)
    gross = np.zeros(n, dtype=np.float64)
    net = np.zeros(n, dtype=np.float64)
    pos = np.zeros(n, dtype=np.float64)
    for i in range(1, n):
        pos[i] = signal[i - 1] if np.isfinite(signal[i - 1]) else 0.0
        r = asset_ret[i] if np.isfinite(asset_ret[i]) else 0.0
        gross[i] = gross[i - 1] + pos[i] * r
        turn = abs(pos[i] - pos[i - 1])
        net[i] = net[i - 1] + pos[i] * r - cost_per_turnover * turn
    return gross, net, pos

@njit
def make_zscore_signal(x, window, direction, entry_z, clip_abs):
    mean, std = rolling_mean_std(x, window)
    n = len(x)
    sig = np.zeros(n, dtype=np.float64)
    for i in range(n):
        if np.isfinite(x[i]) and np.isfinite(mean[i]) and np.isfinite(std[i]) and std[i] > 0:
            z = (x[i] - mean[i]) / std[i]
            if abs(z) >= entry_z:
                raw = direction * z / entry_z
                if raw > clip_abs:
                    raw = clip_abs
                if raw < -clip_abs:
                    raw = -clip_abs
                sig[i] = raw
    return sig

@njit
def slope_r2_tstat(y):
    n = len(y)
    sx = 0.0
    sy = 0.0
    c = 0
    for i in range(n):
        if np.isfinite(y[i]):
            sx += i
            sy += y[i]
            c += 1
    if c < 20:
        return np.nan, np.nan, np.nan
    mx = sx / c
    my = sy / c
    sxx = 0.0
    sxy = 0.0
    syy = 0.0
    for i in range(n):
        if np.isfinite(y[i]):
            dx = i - mx
            dy = y[i] - my
            sxx += dx * dx
            sxy += dx * dy
            syy += dy * dy
    if sxx <= 0 or syy <= 0:
        return np.nan, np.nan, np.nan
    beta = sxy / sxx
    alpha = my - beta * mx
    rss = 0.0
    for i in range(n):
        if np.isfinite(y[i]):
            e = y[i] - alpha - beta * i
            rss += e * e
    r2 = 1.0 - rss / syy
    se = math.sqrt((rss / max(c - 2, 1)) / sxx) if c > 2 else np.nan
    t = beta / se if se > 0 else np.nan
    return beta, r2, t

@njit
def autocorr_lag(x, lag):
    n = len(x)
    sx = 0.0
    sy = 0.0
    c = 0
    for i in range(lag, n):
        if np.isfinite(x[i]) and np.isfinite(x[i-lag]):
            sx += x[i]
            sy += x[i-lag]
            c += 1
    if c < 20:
        return np.nan
    mx = sx / c
    my = sy / c
    vx = 0.0
    vy = 0.0
    cov = 0.0
    for i in range(lag, n):
        if np.isfinite(x[i]) and np.isfinite(x[i-lag]):
            dx = x[i] - mx
            dy = x[i-lag] - my
            vx += dx * dx
            vy += dy * dy
            cov += dx * dy
    if vx <= 0 or vy <= 0:
        return np.nan
    return cov / math.sqrt(vx * vy)

@njit
def detrend_linear_residual(y):
    n = len(y)
    slope, r2, t = slope_r2_tstat(y)
    out = np.empty(n, dtype=np.float64)
    if not np.isfinite(slope):
        for i in range(n):
            out[i] = np.nan
        return out
    sx = 0.0
    sy = 0.0
    c = 0
    for i in range(n):
        if np.isfinite(y[i]):
            sx += i
            sy += y[i]
            c += 1
    if c < 20:
        for i in range(n):
            out[i] = np.nan
        return out
    mx = sx / c
    my = sy / c
    alpha = my - slope * mx
    for i in range(n):
        out[i] = y[i] - alpha - slope * i if np.isfinite(y[i]) else np.nan
    return out

@njit
def half_life_from_series(y):
    n = len(y)
    sx = 0.0
    sy = 0.0
    c = 0
    for i in range(1, n):
        xlag = y[i - 1]
        dx = y[i] - y[i - 1]
        if np.isfinite(xlag) and np.isfinite(dx):
            sx += xlag
            sy += dx
            c += 1
    if c < 30:
        return np.nan, np.nan, np.nan
    mx = sx / c
    my = sy / c
    sxx = 0.0
    sxy = 0.0
    syy = 0.0
    for i in range(1, n):
        xlag = y[i - 1]
        dxv = y[i] - y[i - 1]
        if np.isfinite(xlag) and np.isfinite(dxv):
            a = xlag - mx
            b = dxv - my
            sxx += a * a
            sxy += a * b
            syy += b * b
    if sxx <= 0 or syy <= 0:
        return np.nan, np.nan, np.nan
    gamma = sxy / sxx
    if gamma >= 0 or 1.0 + gamma <= 0:
        return np.nan, gamma, np.nan
    half_life = -math.log(2.0) / math.log(1.0 + gamma)
    alpha = my - gamma * mx
    rss = 0.0
    for i in range(1, n):
        xlag = y[i - 1]
        dxv = y[i] - y[i - 1]
        if np.isfinite(xlag) and np.isfinite(dxv):
            e = dxv - alpha - gamma * xlag
            rss += e * e
    r2 = 1.0 - rss / syy
    return half_life, gamma, r2

@njit
def rolling_half_life_summary(y, window, step):
    n = len(y)
    vals = np.empty(n, dtype=np.float64)
    count = 0
    for end in range(window, n + 1, step):
        sub = y[end-window:end]
        hl, gamma, r2 = half_life_from_series(sub)
        if np.isfinite(hl):
            vals[count] = hl
            count += 1
    if count < 3:
        return np.nan, np.nan, np.nan, 0
    for i in range(count):
        for j in range(i + 1, count):
            if vals[j] < vals[i]:
                tmp = vals[i]
                vals[i] = vals[j]
                vals[j] = tmp
    median = vals[count // 2] if count % 2 == 1 else 0.5 * (vals[count // 2 - 1] + vals[count // 2])
    p25 = vals[int(math.floor((count - 1) * 0.25))]
    p75 = vals[int(math.floor((count - 1) * 0.75))]
    return median, p25, p75, count

## Core Strategies (базовые стратегии)

Мы строим несколько типов:

- `spread_mean_reversion (возврат спреда к среднему)`;
- `spread_momentum (моментум спреда)`;
- `pair_trend_portfolio (тренд портфеля двух ног)`;
- `anti_corr_pair_trend (тренд анти-коррелированной пары)`;
- `sector_trend (секторный тренд)`;
- `inverse_* (инверсия стратегии)` для явного поиска отрицательных, но структурных `PnL (прибыль и убыток)`.

In [60]:
def beta_ols_log(y, x):
    tmp = pd.concat([np.log(y).rename('y'), np.log(x).rename('x')], axis=1).dropna()
    if len(tmp) < 200:
        return np.nan, np.nan
    vx = tmp['x'].var()
    if vx <= 0:
        return np.nan, np.nan
    beta = tmp['y'].cov(tmp['x']) / vx
    alpha = tmp['y'].mean() - beta * tmp['x'].mean()
    return float(alpha), float(beta)

def build_pair_spread(panel, y, x):
    px = panel[[y, x]].dropna()
    alpha, beta = beta_ols_log(filter_by_split(px, 'train')[y], filter_by_split(px, 'train')[x])
    log_px = np.log(px)
    spread = log_px[y] - alpha - beta * log_px[x]
    spread_ret = spread.diff().fillna(0)
    return spread, spread_ret, alpha, beta

candidate_pairs = pair_stats[['group','y','x']].drop_duplicates().copy()

extra_negative = negative_cross_pairs.head(12).rename(columns={'a':'y','b':'x'})
extra_negative['group'] = 'negative_cross_sector'
extra_negative = extra_negative[['group','y','x']]

strategy_pairs = pd.concat([candidate_pairs, extra_negative], ignore_index=True).drop_duplicates(['y','x'])
strategy_pairs.to_csv(OUTPUT_DIR / '05_strategy_pair_universe.csv', index=False)
strategy_pairs.head(30)

,group,y,x
0,energy_cracks,RB,CL
1,grain_complex,ZW,KE
2,soy_crush_and_oilseeds,ZM,ZS
3,energy_cracks,RB,HO
4,livestock,LE,GF
5,grain_complex,ZC,ZW
6,grain_complex,ZC,ZS
7,soy_crush_and_oilseeds,ZL,ZS
8,grain_complex,KE,ZC
9,energy_cracks,HO,CL


In [61]:
def split_metrics(equity, net_equity, pos, split_name):
    eq = filter_by_split(equity, split_name).dropna()
    neq = filter_by_split(net_equity, split_name).dropna()
    p = filter_by_split(pos, split_name).reindex(neq.index).fillna(0)
    if len(eq) < 30:
        return {}
    gap = (eq.reindex(neq.index) - neq).dropna()
    deq = eq.diff().dropna()
    dneq = neq.diff().dropna()
    slope, trend_r2, slope_t = slope_r2_tstat(neq.to_numpy(dtype=np.float64) - float(neq.iloc[0]))
    gap_slope, gap_trend_r2, gap_slope_t = slope_r2_tstat(gap.to_numpy(dtype=np.float64) - float(gap.iloc[0])) if len(gap) >= 30 else (np.nan, np.nan, np.nan)
    neq_detrended = detrend_linear_residual(neq.to_numpy(dtype=np.float64))
    half_life_bars, half_life_gamma, half_life_r2 = half_life_from_series(neq_detrended)
    rolling_hl_window = int(min(max(500, len(neq) // 4), 4000))
    rolling_hl_step = int(max(50, rolling_hl_window // 10))
    rolling_hl_median, rolling_hl_p25, rolling_hl_p75, rolling_hl_n = rolling_half_life_summary(
        neq_detrended,
        rolling_hl_window,
        rolling_hl_step,
    )
    gross_total = float(eq.iloc[-1] - eq.iloc[0])
    net_total = float(neq.iloc[-1] - neq.iloc[0])
    gap_total = float(gap.iloc[-1] - gap.iloc[0]) if len(gap) else np.nan
    sign = np.sign(float(neq.iloc[-1] - neq.iloc[0]))
    sign_consistency = float((np.sign(dneq) == sign).mean()) if sign != 0 and len(dneq) else np.nan
    running_max = neq.cummax()
    dd = neq - running_max
    turnover = p.diff().abs().sum()
    return {
        f'{split_name}_gross_total': gross_total,
        f'{split_name}_net_total': net_total,
        f'{split_name}_cost_gap_total': gap_total,
        f'{split_name}_cost_gap_to_abs_gross': abs(gap_total) / abs(gross_total) if gross_total != 0 and np.isfinite(gap_total) else np.nan,
        f'{split_name}_cost_gap_to_abs_net': abs(gap_total) / abs(net_total) if net_total != 0 and np.isfinite(gap_total) else np.nan,
        f'{split_name}_cost_gap_slope': gap_slope,
        f'{split_name}_cost_gap_trend_r2': gap_trend_r2,
        f'{split_name}_cost_gap_slope_t': gap_slope_t,
        f'{split_name}_detrended_half_life_bars': half_life_bars,
        f'{split_name}_detrended_half_life_hours': half_life_bars * 0.5 if np.isfinite(half_life_bars) else np.nan,
        f'{split_name}_detrended_half_life_gamma': half_life_gamma,
        f'{split_name}_detrended_half_life_r2': half_life_r2,
        f'{split_name}_rolling_half_life_median_bars': rolling_hl_median,
        f'{split_name}_rolling_half_life_p25_bars': rolling_hl_p25,
        f'{split_name}_rolling_half_life_p75_bars': rolling_hl_p75,
        f'{split_name}_rolling_half_life_n_windows': int(rolling_hl_n),
        f'{split_name}_adaptive_T2_bars': int(np.clip(round(rolling_hl_median), 4, 1500)) if np.isfinite(rolling_hl_median) else np.nan,
        f'{split_name}_net_slope': slope,
        f'{split_name}_net_trend_r2': trend_r2,
        f'{split_name}_net_slope_t': slope_t,
        f'{split_name}_sign_consistency': sign_consistency,
        f'{split_name}_pnl_autocorr_1': autocorr_lag(dneq.to_numpy(dtype=np.float64), 1),
        f'{split_name}_pnl_autocorr_5': autocorr_lag(dneq.to_numpy(dtype=np.float64), 5),
        f'{split_name}_max_drawdown': float(dd.min()),
        f'{split_name}_turnover': float(turnover),
        f'{split_name}_n_bars': int(len(neq)),
    }

def monthly_seasonality_score(net_equity, split_name):
    eq = filter_by_split(net_equity, split_name).dropna()
    if len(eq) < 500:
        return np.nan, np.nan
    daily = eq.resample('1D').last().dropna().diff().dropna()
    if len(daily) < 60:
        return np.nan, np.nan
    by_month = daily.groupby(daily.index.month)
    means = by_month.mean()
    counts = by_month.count()
    if len(means) < 6 or daily.std() == 0:
        return np.nan, np.nan
    strength = float((means.max() - means.min()) / daily.std())
    top3_share = float(means.sort_values(ascending=False).head(3).sum() / abs(means).sum()) if abs(means).sum() > 0 else np.nan
    return strength, top3_share

def classify_geometry(row):
    slope_t = row.get('train_net_slope_t', np.nan)
    r2 = row.get('train_net_trend_r2', np.nan)
    ac1 = row.get('train_pnl_autocorr_1', np.nan)
    seas = row.get('train_monthly_seasonality_strength', np.nan)
    total = row.get('train_net_total', np.nan)
    if np.isfinite(slope_t) and np.isfinite(r2) and abs(slope_t) >= 2 and r2 >= 0.25:
        return 'trend_up (тренд вверх)' if total > 0 else 'trend_down (тренд вниз)'
    if np.isfinite(ac1) and ac1 <= -0.05:
        return 'mean_reverting_pnl (возвратный PnL)'
    if np.isfinite(seas) and seas >= 0.75:
        return 'seasonal_pnl (сезонный PnL)'
    if np.isfinite(ac1) and ac1 >= 0.05:
        return 'serial_pnl (серийный PnL)'
    return 'noisy_or_weak (шумный или слабый)'

In [62]:
strategy_rows = []

WINDOWS = [44, 88, 176]
ENTRY_Z = 1.0
CLIP_ABS = 1.0

for _, pr in strategy_pairs.iterrows():
    y, x, source_group = pr['y'], pr['x'], pr['group']
    if y not in panel_30m.columns or x not in panel_30m.columns:
        continue
    try:
        spread, spread_ret, alpha, beta = build_pair_spread(panel_30m, y, x)
    except Exception:
        continue
    for window in WINDOWS:
        spread_arr = spread.to_numpy(dtype=np.float64)
        spread_ret_arr = spread_ret.reindex(spread.index).fillna(0).to_numpy(dtype=np.float64)
        for family, direction in [('spread_mean_reversion', -1.0), ('spread_momentum', 1.0)]:
            sig = make_zscore_signal(spread_arr, window, direction, ENTRY_Z, CLIP_ABS)
            for inverse in [False, True]:
                use_sig = -sig if inverse else sig
                g, n, pos_arr = strategy_from_signal(spread_ret_arr, use_sig, COST_PER_UNIT_TURNOVER)
                p = pd.Series(pos_arr, index=spread.index)
                name = ('inverse_' if inverse else '') + family
                eq = pd.Series(g, index=spread.index)
                neq = pd.Series(n, index=spread.index)
                row = {'source_group': source_group, 'y': y, 'x': x, 'strategy_family': name, 'window': window, 'alpha': alpha, 'beta': beta}
                for split_name in splits:
                    row.update(split_metrics(eq, neq, p, split_name))
                    seas, top3 = monthly_seasonality_score(neq, split_name)
                    row[f'{split_name}_monthly_seasonality_strength'] = seas
                    row[f'{split_name}_top3_month_share'] = top3
                row['geometry_class'] = classify_geometry(row)
                strategy_rows.append(row)

# Sector trend portfolios.
for group, r in group_returns.items():
    idx = r.dropna().index
    ret = r.reindex(idx).fillna(0).to_numpy(dtype=np.float64)
    cumulative = np.cumsum(ret)
    for window in WINDOWS:
        for direction, fam in [(1.0, 'sector_trend'), (-1.0, 'inverse_sector_trend')]:
            sig = make_zscore_signal(cumulative, window, direction, ENTRY_Z, CLIP_ABS)
            gross, net, pos_arr = strategy_from_signal(ret, sig, COST_PER_UNIT_TURNOVER)
            eq = pd.Series(gross, index=idx)
            neq = pd.Series(net, index=idx)
            p = pd.Series(pos_arr, index=idx)
            row = {'source_group': group, 'y': group, 'x': 'sector_index', 'strategy_family': fam, 'window': window, 'alpha': np.nan, 'beta': np.nan}
            for split_name in splits:
                row.update(split_metrics(eq, neq, p, split_name))
                seas, top3 = monthly_seasonality_score(neq, split_name)
                row[f'{split_name}_monthly_seasonality_strength'] = seas
                row[f'{split_name}_top3_month_share'] = top3
            row['geometry_class'] = classify_geometry(row)
            strategy_rows.append(row)

geometry = pd.DataFrame(strategy_rows)
geometry['calibration_test_net_sum'] = geometry.get('calibration_net_total', 0) + geometry.get('test_net_total', 0)
geometry['calibration_test_gross_sum'] = geometry.get('calibration_gross_total', 0) + geometry.get('test_gross_total', 0)
geometry['calibration_test_cost_gap_sum'] = geometry.get('calibration_cost_gap_total', 0) + geometry.get('test_cost_gap_total', 0)
geometry['all_split_cost_gap_sum'] = geometry.get('train_cost_gap_total', 0) + geometry.get('calibration_cost_gap_total', 0) + geometry.get('test_cost_gap_total', 0)
geometry['all_split_abs_gross_sum'] = (
    geometry.get('train_gross_total', 0).abs()
    + geometry.get('calibration_gross_total', 0).abs()
    + geometry.get('test_gross_total', 0).abs()
)
geometry['cost_gap_to_abs_gross_all'] = geometry['all_split_cost_gap_sum'].abs() / geometry['all_split_abs_gross_sum'].replace(0, np.nan)
geometry['calibration_test_cost_gap_to_abs_gross'] = geometry['calibration_test_cost_gap_sum'].abs() / geometry['calibration_test_gross_sum'].abs().replace(0, np.nan)
geometry['cost_dominated_all'] = geometry['cost_gap_to_abs_gross_all'] >= 0.50
geometry['cost_dominated_calibration_test'] = geometry['calibration_test_cost_gap_to_abs_gross'] >= 0.50
geometry['all_splits_same_sign'] = (
    np.sign(geometry.get('train_net_total')) == np.sign(geometry.get('calibration_net_total'))
) & (
    np.sign(geometry.get('train_net_total')) == np.sign(geometry.get('test_net_total'))
) & (geometry.get('train_net_total').abs() > 0)
geometry = geometry.sort_values(['geometry_class','all_splits_same_sign','calibration_test_net_sum'], ascending=[True, False, False])
geometry.to_csv(OUTPUT_DIR / '06_core_strategy_geometry_metrics.csv', index=False)

cost_cols = [
    'source_group', 'y', 'x', 'strategy_family', 'window', 'geometry_class',
    'train_gross_total', 'train_net_total', 'train_cost_gap_total', 'train_cost_gap_to_abs_gross', 'train_cost_gap_trend_r2',
    'calibration_gross_total', 'calibration_net_total', 'calibration_cost_gap_total', 'calibration_cost_gap_to_abs_gross', 'calibration_cost_gap_trend_r2',
    'test_gross_total', 'test_net_total', 'test_cost_gap_total', 'test_cost_gap_to_abs_gross', 'test_cost_gap_trend_r2',
    'calibration_test_gross_sum', 'calibration_test_net_sum', 'calibration_test_cost_gap_sum',
    'cost_gap_to_abs_gross_all', 'calibration_test_cost_gap_to_abs_gross',
    'cost_dominated_all', 'cost_dominated_calibration_test',
    'train_turnover', 'calibration_turnover', 'test_turnover',
]
cost_summary = geometry[[c for c in cost_cols if c in geometry.columns]].sort_values(
    ['cost_dominated_calibration_test', 'calibration_test_cost_gap_to_abs_gross', 'calibration_test_net_sum'],
    ascending=[True, True, False],
)
cost_summary.to_csv(OUTPUT_DIR / '06b_cost_gap_summary.csv', index=False)

hl_cols = [
    'source_group', 'y', 'x', 'strategy_family', 'window', 'geometry_class',
    'train_detrended_half_life_bars', 'train_detrended_half_life_hours',
    'train_detrended_half_life_gamma', 'train_detrended_half_life_r2',
    'train_rolling_half_life_median_bars', 'train_rolling_half_life_p25_bars', 'train_rolling_half_life_p75_bars',
    'train_rolling_half_life_n_windows', 'train_adaptive_T2_bars',
    'calibration_detrended_half_life_bars', 'calibration_rolling_half_life_median_bars', 'calibration_adaptive_T2_bars',
    'test_detrended_half_life_bars', 'test_rolling_half_life_median_bars', 'test_adaptive_T2_bars',
    'train_pnl_autocorr_1', 'calibration_pnl_autocorr_1', 'test_pnl_autocorr_1',
    'train_net_trend_r2', 'calibration_net_trend_r2', 'test_net_trend_r2',
    'calibration_test_net_sum', 'calibration_test_cost_gap_to_abs_gross',
]
half_life_summary = geometry[[c for c in hl_cols if c in geometry.columns]].copy()
half_life_summary['half_life_available'] = np.isfinite(half_life_summary['train_detrended_half_life_bars'])
half_life_summary['half_life_stable_train_calibration'] = (
    np.isfinite(half_life_summary['train_rolling_half_life_median_bars'])
    & np.isfinite(half_life_summary['calibration_rolling_half_life_median_bars'])
    & (
        (
            half_life_summary['train_rolling_half_life_median_bars']
            / half_life_summary['calibration_rolling_half_life_median_bars'].replace(0, np.nan)
        ).between(0.5, 2.0)
    )
)
half_life_summary = half_life_summary.sort_values(
    ['half_life_available', 'half_life_stable_train_calibration', 'train_detrended_half_life_bars'],
    ascending=[False, False, True],
)
half_life_summary.to_csv(OUTPUT_DIR / '06c_half_life_diagnostics.csv', index=False)

geometry.head(25)

,source_group,y,x,strategy_family,window,alpha,beta,train_gross_total,train_net_total,train_cost_gap_total,...,calibration_test_net_sum,calibration_test_gross_sum,calibration_test_cost_gap_sum,all_split_cost_gap_sum,all_split_abs_gross_sum,cost_gap_to_abs_gross_all,calibration_test_cost_gap_to_abs_gross,cost_dominated_all,cost_dominated_calibration_test,all_splits_same_sign
244,softs_low_priority,softs_low_priority,sector_index,sector_trend,176,NaN,NaN,0.285793,0.279943,0.00585,...,0.210224,0.214524,0.00430,0.01015,0.709346,0.014309,0.020044,False,False,False
242,softs_low_priority,softs_low_priority,sector_index,sector_trend,88,NaN,NaN,0.213227,0.203777,0.00945,...,0.015523,0.023673,0.00815,0.01760,0.745316,0.023614,0.344279,False,False,False
243,softs_low_priority,softs_low_priority,sector_index,inverse_sector_trend,88,NaN,NaN,-0.213227,-0.222677,0.00945,...,-0.031823,-0.023673,0.00815,0.01760,0.745316,0.023614,0.344279,False,False,False
188,fx_usd_factor,DX,E6,spread_mean_reversion,176,4.658251,-0.907108,2.127794,1.652194,0.47560,...,-0.042165,0.290735,0.33290,0.80850,2.418529,0.334294,1.145030,False,True,False
191,fx_usd_factor,DX,E6,inverse_spread_momentum,176,4.658251,-0.907108,2.127794,1.652194,0.47560,...,-0.042165,0.290735,0.33290,0.80850,2.418529,0.334294,1.145030,False,True,False
184,fx_usd_factor,DX,E6,spread_mean_reversion,88,4.658251,-0.907108,2.510470,1.861970,0.64850,...,-0.115416,0.366384,0.48180,1.13030,2.876853,0.392895,1.315014,False,True,False
187,fx_usd_factor,DX,E6,inverse_spread_momentum,88,4.658251,-0.907108,2.510470,1.861970,0.64850,...,-0.115416,0.366384,0.48180,1.13030,2.876853,0.392895,1.315014,False,True,False
180,fx_usd_factor,DX,E6,spread_mean_reversion,44,4.658251,-0.907108,3.022869,2.085819,0.93705,...,-0.161849,0.529251,0.69110,1.62815,3.552121,0.458360,1.305807,False,True,False
183,fx_usd_factor,DX,E6,inverse_spread_momentum,44,4.658251,-0.907108,3.022869,2.085819,0.93705,...,-0.161849,0.529251,0.69110,1.62815,3.552121,0.458360,1.305807,False,True,False
245,softs_low_priority,softs_low_priority,sector_index,inverse_sector_trend,176,NaN,NaN,-0.285793,-0.291643,0.00585,...,-0.218824,-0.214524,0.00430,0.01015,0.709346,0.014309,0.020044,False,False,False


In [63]:
summary = geometry.groupby(['geometry_class','strategy_family']).agg(
    n=('strategy_family','count'),
    same_sign=('all_splits_same_sign','sum'),
    median_train_net=('train_net_total','median'),
    median_calibration_net=('calibration_net_total','median'),
    median_test_net=('test_net_total','median'),
    median_train_slope_t=('train_net_slope_t','median'),
    median_train_ac1=('train_pnl_autocorr_1','median'),
    median_seasonality=('train_monthly_seasonality_strength','median'),
).reset_index().sort_values(['same_sign','median_test_net'], ascending=[False, False])
summary.to_csv(OUTPUT_DIR / '08_geometry_class_summary.csv', index=False)
summary

,geometry_class,strategy_family,n,same_sign,median_train_net,median_calibration_net,median_test_net,median_train_slope_t,median_train_ac1,median_seasonality
11,trend_down (тренд вниз),inverse_spread_mean_reversion,52,51,-3.654373,-0.568788,-0.782346,-1021.541154,-0.051630,0.263331
15,trend_down (тренд вниз),spread_momentum,52,51,-3.654373,-0.568788,-0.782346,-1021.541154,-0.051630,0.263331
17,trend_up (тренд вверх),inverse_spread_momentum,41,39,2.938750,0.395009,0.517799,755.075748,-0.070247,0.251340
19,trend_up (тренд вверх),spread_mean_reversion,41,39,2.938750,0.395009,0.517799,755.075748,-0.070247,0.251340
4,noisy_or_weak (шумный или слабый),inverse_sector_trend,7,2,0.003638,0.191042,-0.314236,-14.652468,-0.019689,0.250779
7,noisy_or_weak (шумный или слабый),sector_trend,6,1,0.000411,-0.161822,0.512544,13.566796,-0.022532,0.262360
13,trend_down (тренд вниз),sector_trend,2,1,-0.488251,-0.203679,0.010358,-39.047623,-0.005031,0.187143
6,noisy_or_weak (шумный или слабый),inverse_spread_momentum,5,1,0.522959,0.069492,-0.086447,93.475715,-0.039529,0.516919
8,noisy_or_weak (шумный или слабый),spread_mean_reversion,5,1,0.522959,0.069492,-0.086447,93.475715,-0.039529,0.516919
12,trend_down (тренд вниз),inverse_spread_momentum,5,1,-0.489276,0.109903,-0.145341,-462.459316,-0.027433,0.228416


In [64]:
def stable_strategy_id(row):
    if row['x'] == 'sector_index':
        return f"sector_{row['y']}_{row['strategy_family']}_{int(row['window'])}"
    return f"{row['source_group']}_{row['y']}_{row['x']}_{row['strategy_family']}_{int(row['window'])}"

geometry['strategy_id'] = geometry.apply(stable_strategy_id, axis=1)

positive_mask = (
    (geometry['train_net_total'] > 0)
    & (geometry['calibration_net_total'] > 0)
    & (geometry['test_net_total'] > 0)
)
negative_mask = (
    (geometry['train_net_total'] < 0)
    & (geometry['calibration_net_total'] < 0)
    & (geometry['test_net_total'] < 0)
)

def select_without_duplicates(df, sort_cols, ascending, max_rows, used_ids):
    out = df[~df['strategy_id'].isin(used_ids)].copy()
    if out.empty:
        return out
    out = out.sort_values(sort_cols, ascending=ascending).head(max_rows)
    used_ids.update(out['strategy_id'].tolist())
    return out

used_strategy_ids = set()
plot_groups = {}

seasonality_score = geometry[['train_monthly_seasonality_strength', 'calibration_monthly_seasonality_strength', 'test_monthly_seasonality_strength']].mean(axis=1)
mean_rev_or_seasonal = geometry[
    geometry['geometry_class'].isin(['mean_reverting_pnl (возвратный PnL)', 'seasonal_pnl (сезонный PnL)', 'serial_pnl (серийный PnL)'])
    | (seasonality_score >= seasonality_score.quantile(0.80))
].copy()
mean_rev_or_seasonal['seasonality_score'] = seasonality_score.reindex(mean_rev_or_seasonal.index)
mean_rev_or_seasonal['abs_calibration_test_net_sum'] = mean_rev_or_seasonal['calibration_test_net_sum'].abs()

# Shape-first bucket: keep mean-reverting/seasonal/serial PnL (прибыль и убыток) even if final PnL is not sign-persistent.
plot_groups['mean_reversion_seasonality_all'] = select_without_duplicates(
    mean_rev_or_seasonal,
    ['geometry_class', 'seasonality_score', 'abs_calibration_test_net_sum'],
    [True, False, False],
    36,
    used_strategy_ids,
)

# Remaining consistently losing strategies are saved as negative PnL (отрицательный PnL) candidates.
plot_groups['negative_pnl'] = select_without_duplicates(
    geometry[negative_mask],
    ['calibration_test_net_sum', 'train_net_trend_r2'],
    [True, False],
    200,
    used_strategy_ids,
)

# Positive trend groups are mutually exclusive and exclude already selected shape-first candidates.
plot_groups['trend_up_positive'] = select_without_duplicates(
    geometry[positive_mask & (geometry['geometry_class'] == 'trend_up (тренд вверх)')],
    ['calibration_test_net_sum', 'test_net_trend_r2', 'train_net_trend_r2'],
    [False, False, False],
    36,
    used_strategy_ids,
)

plot_groups['trend_down_positive'] = select_without_duplicates(
    geometry[positive_mask & (geometry['geometry_class'] == 'trend_down (тренд вниз)')],
    ['calibration_test_net_sum', 'test_net_trend_r2', 'train_net_trend_r2'],
    [False, False, False],
    36,
    used_strategy_ids,
)

candidate_rows = []
for group_name, candidates in plot_groups.items():
    tmp = candidates.copy()
    tmp.insert(0, 'plot_group', group_name)
    candidate_rows.append(tmp)

plot_candidates = pd.concat(candidate_rows, ignore_index=True) if candidate_rows else pd.DataFrame()
plot_candidates.to_csv(OUTPUT_DIR / '09_plot_candidate_groups.csv', index=False)

def rebuild_strategy_path(row):
    family = row['strategy_family']
    window = int(row['window'])
    y = row['y']
    x = row['x']
    if x == 'sector_index':
        idx = group_returns[y].dropna().index
        ret = group_returns[y].reindex(idx).fillna(0).to_numpy(dtype=np.float64)
        cumulative = np.cumsum(ret)
        direction = -1.0 if family.startswith('inverse_') else 1.0
        sig = make_zscore_signal(cumulative, window, direction, ENTRY_Z, CLIP_ABS)
        gross, net, pos_arr = strategy_from_signal(ret, sig, COST_PER_UNIT_TURNOVER)
        sid = f"sector_{y}_{family}_{window}"
        return sid, pd.DataFrame({'gross_pnl': gross, 'net_pnl': net, 'position': pos_arr}, index=idx)

    spread, spread_ret, alpha, beta = build_pair_spread(panel_30m, y, x)
    direction = -1.0 if 'mean_reversion' in family else 1.0
    sig = make_zscore_signal(spread.to_numpy(dtype=np.float64), window, direction, ENTRY_Z, CLIP_ABS)
    if family.startswith('inverse_'):
        sig = -sig
    gross, net, pos_arr = strategy_from_signal(spread_ret.reindex(spread.index).fillna(0).to_numpy(dtype=np.float64), sig, COST_PER_UNIT_TURNOVER)
    sid = f"{row['source_group']}_{y}_{x}_{family}_{window}"
    return sid, pd.DataFrame({'gross_pnl': gross, 'net_pnl': net, 'position': pos_arr}, index=spread.index)

fig_rows = []
path_rows = []

def downsample_path_for_plot(pth, max_points=MAX_PLOT_POINTS):
    if len(pth) <= max_points:
        return pth.copy()
    take = np.linspace(0, len(pth) - 1, max_points).astype(np.int64)
    return pth.iloc[take].copy()

for group_name, candidates in plot_groups.items():
    group_dir = FIG_DIR / group_name
    path_group_dir = PATH_DIR / group_name
    group_dir.mkdir(parents=True, exist_ok=True)
    path_group_dir.mkdir(parents=True, exist_ok=True)
    for _, row in candidates.iterrows():
        sid, pth = rebuild_strategy_path(row)
        if pth.empty:
            continue
        pth = pth.sort_index()
        plot_pth = downsample_path_for_plot(pth)
        if SAVE_DOWNSAMPLED_PNL_PATHS:
            path_path = path_group_dir / f"{sid}.parquet"
            plot_pth.assign(Date=plot_pth.index, strategy_id=sid, plot_group=group_name).reset_index(drop=True).to_parquet(path_path, index=False)
        else:
            path_path = None

        # One figure contains exactly one core strategy (базовая стратегия).
        # The two lines compare gross PnL (валовый PnL) and net PnL (чистый PnL) for that same strategy.
        plot_pth['cost_gap'] = plot_pth['gross_pnl'] - plot_pth['net_pnl']
        fig, axes = plt.subplots(3, 1, figsize=(13, 7.5), sharex=True, gridspec_kw={'height_ratios': [2.2, 1.0, 1.0]})
        axes[0].plot(plot_pth.index, plot_pth['gross_pnl'], label='gross PnL (валовый PnL)', linewidth=1.1)
        axes[0].plot(plot_pth.index, plot_pth['net_pnl'], label='net PnL (чистый PnL)', alpha=0.85, linewidth=1.1)
        for split_name, spec in splits.items():
            axes[0].axvspan(spec['start'], spec['end'], alpha=0.06)
        axes[0].set_title(f"{sid}: {row['geometry_class']} | {group_name}")
        axes[0].legend(loc='best')
        axes[1].plot(plot_pth.index, plot_pth['cost_gap'], label='gross-net cost gap (разрыв издержек)', color='tab:red', linewidth=1.0)
        axes[1].legend(loc='best')
        axes[2].plot(plot_pth.index, plot_pth['position'], label='position (позиция)', color='tab:purple', linewidth=0.9)
        axes[2].legend(loc='best')
        fig.tight_layout()
        fig_path = group_dir / f"geometry_{sid}.png"
        fig.savefig(fig_path, dpi=110)
        plt.close(fig)
        fig_rows.append({
            'plot_group': group_name,
            'strategy_id': sid,
            'figure': str(fig_path),
            'geometry_class': row['geometry_class'],
            'train_net_total': row['train_net_total'],
            'calibration_net_total': row['calibration_net_total'],
            'test_net_total': row['test_net_total'],
            'calibration_test_net_sum': row['calibration_test_net_sum'],
            'train_detrended_half_life_bars': row.get('train_detrended_half_life_bars', np.nan),
            'train_rolling_half_life_median_bars': row.get('train_rolling_half_life_median_bars', np.nan),
            'train_adaptive_T2_bars': row.get('train_adaptive_T2_bars', np.nan),
        })
        if path_path is not None:
            path_rows.append({
                'plot_group': group_name,
                'strategy_id': sid,
                'path_file': str(path_path),
                'saved_rows': len(plot_pth),
                'full_rows': len(pth),
                'train_adaptive_T2_bars': row.get('train_adaptive_T2_bars', np.nan),
            })

    if not candidates.empty:
        candidates.to_csv(OUTPUT_DIR / f"09_{group_name}_candidates.csv", index=False)

figures = pd.DataFrame(fig_rows)
figures.to_csv(OUTPUT_DIR / '09_geometry_figures.csv', index=False)
path_index = pd.DataFrame(path_rows)
path_index.to_csv(OUTPUT_DIR / '07_selected_core_strategy_pnl_path_index.csv', index=False)
print('saved figures:', len(figures), '| saved per-strategy path files:', len(path_index))
figures.head(20)

saved figures: 164 | saved per-strategy path files: 164


,plot_group,strategy_id,figure,geometry_class,train_net_total,calibration_net_total,test_net_total,calibration_test_net_sum,train_detrended_half_life_bars,train_rolling_half_life_median_bars,train_adaptive_T2_bars
0,mean_reversion_seasonality_all,fx_usd_factor_DX_E6_spread_mean_reversion_176,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,mean_reverting_pnl (возвратный PnL),1.652194,-0.007732,-0.034433,-0.042165,2185.197688,787.031767,787
1,mean_reversion_seasonality_all,fx_usd_factor_DX_E6_inverse_spread_momentum_176,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,mean_reverting_pnl (возвратный PnL),1.652194,-0.007732,-0.034433,-0.042165,2185.197688,787.031767,787
2,mean_reversion_seasonality_all,fx_usd_factor_DX_E6_spread_mean_reversion_44,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,mean_reverting_pnl (возвратный PnL),2.085819,-0.077508,-0.084341,-0.161849,2261.156265,1231.292715,1231
3,mean_reversion_seasonality_all,fx_usd_factor_DX_E6_inverse_spread_momentum_44,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,mean_reverting_pnl (возвратный PnL),2.085819,-0.077508,-0.084341,-0.161849,2261.156265,1231.292715,1231
4,mean_reversion_seasonality_all,fx_usd_factor_DX_E6_spread_mean_reversion_88,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,mean_reverting_pnl (возвратный PnL),1.861970,-0.036553,-0.078863,-0.115416,2275.774532,1033.676134,1034
5,mean_reversion_seasonality_all,fx_usd_factor_DX_E6_inverse_spread_momentum_88,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,mean_reverting_pnl (возвратный PnL),1.861970,-0.036553,-0.078863,-0.115416,2275.774532,1033.676134,1034
6,mean_reversion_seasonality_all,sector_softs_low_priority_sector_trend_176,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,mean_reverting_pnl (возвратный PnL),0.279943,-0.106815,0.317039,0.210224,98.344524,34.721108,35
7,mean_reversion_seasonality_all,sector_softs_low_priority_inverse_sector_trend...,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,mean_reverting_pnl (возвратный PnL),-0.291643,0.102215,-0.321039,-0.218824,97.877420,34.500713,35
8,mean_reversion_seasonality_all,sector_softs_low_priority_sector_trend_88,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,mean_reverting_pnl (возвратный PnL),0.203777,-0.258608,0.274131,0.015523,134.960203,34.547892,35
9,mean_reversion_seasonality_all,sector_softs_low_priority_inverse_sector_trend_88,C:\Users\jack\Jupyter_files\QUANT_TRAIDING\Fit...,mean_reverting_pnl (возвратный PnL),-0.222677,0.249808,-0.281631,-0.031823,134.360358,34.341757,34


In [65]:
readme = '''# core_strategy_geometry_audit

Outputs:

- `00_protocol_splits.csv` - train/calibration/test split (разбиение на обучение/калибровку/тест).
- `01_loaded_panel_meta.csv` - market panel metadata (мета-информация панели рынков).
- `02_sector_return_correlation.csv` - sector correlation matrix (матрица корреляции секторов).
- `03_all_market_daily_correlation_pairs.csv` - all daily market correlations (все дневные корреляции рынков).
- `04_negative_cross_sector_pairs.csv` - negative cross-sector pairs (отрицательно коррелированные межсекторные пары).
- `05_strategy_pair_universe.csv` - pairs used for strategy generation (пары для генерации стратегий).
- `06_core_strategy_geometry_metrics.csv` - core strategy geometry metrics (метрики геометрии базовых стратегий).
- `06b_cost_gap_summary.csv` - gross/net PnL cost gap diagnostics (диагностика разрыва между валовым и чистым PnL из-за издержек).
- `06c_half_life_diagnostics.csv` - detrended and rolling half-life diagnostics for PnL (диагностика периода полураспада очищенного от тренда и скользящего PnL).
- `07_selected_core_strategy_pnl_path_index.csv` - index of per-strategy downsampled PnL path files (индекс отдельных прореженных траекторий PnL по стратегиям).
- `08_geometry_class_summary.csv` - summary by geometry class (сводка по классу геометрии).
- `09_plot_candidate_groups.csv` - strategy-to-plot-group mapping (соответствие стратегии и группы графиков).
- `09_*_candidates.csv` - candidates saved for each plot group (кандидаты для каждой группы графиков).
- `09_geometry_figures.csv` and `figures/*/*.png` - saved grouped charts (сохранённые графики по группам).
- `paths/*/*.parquet` - one downsampled PnL path per plotted strategy (одна прореженная траектория PnL на одну нарисованную стратегию).

Interpretation: this notebook deliberately includes inverse and trend variants, so negative or non-classical core strategies can be selected if their PnL shape is exploitable by a second-layer option filter.
Plot grouping is shape-first: `mean_reversion_seasonality_all` ignores final PnL sign (знак итогового PnL), while `negative_pnl` contains remaining consistently losing strategies.
'''
(OUTPUT_DIR / 'README.md').write_text(readme, encoding='utf-8')

manifest = {
    'source_panel': str(PANEL_PATH.resolve()),
    'protocol_file': str(PROTOCOL_PATH.resolve()),
    'protocol': protocol_cfg,
    'has_numba': HAS_NUMBA,
    'cost_per_unit_turnover': COST_PER_UNIT_TURNOVER,
    'saved_files': sorted([str(p.relative_to(OUTPUT_DIR)) for p in OUTPUT_DIR.rglob('*') if p.is_file()]),
}
with (OUTPUT_DIR / '00_manifest.json').open('w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2, default=str)

print('Saved to:', OUTPUT_DIR.resolve())
summary.head(20)

Saved to: ./data\core_strategy_geometry_audit


,geometry_class,strategy_family,n,same_sign,median_train_net,median_calibration_net,median_test_net,median_train_slope_t,median_train_ac1,median_seasonality
11,trend_down (тренд вниз),inverse_spread_mean_reversion,52,51,-3.654373,-0.568788,-0.782346,-1021.541154,-0.051630,0.263331
15,trend_down (тренд вниз),spread_momentum,52,51,-3.654373,-0.568788,-0.782346,-1021.541154,-0.051630,0.263331
17,trend_up (тренд вверх),inverse_spread_momentum,41,39,2.938750,0.395009,0.517799,755.075748,-0.070247,0.251340
19,trend_up (тренд вверх),spread_mean_reversion,41,39,2.938750,0.395009,0.517799,755.075748,-0.070247,0.251340
4,noisy_or_weak (шумный или слабый),inverse_sector_trend,7,2,0.003638,0.191042,-0.314236,-14.652468,-0.019689,0.250779
7,noisy_or_weak (шумный или слабый),sector_trend,6,1,0.000411,-0.161822,0.512544,13.566796,-0.022532,0.262360
13,trend_down (тренд вниз),sector_trend,2,1,-0.488251,-0.203679,0.010358,-39.047623,-0.005031,0.187143
6,noisy_or_weak (шумный или слабый),inverse_spread_momentum,5,1,0.522959,0.069492,-0.086447,93.475715,-0.039529,0.516919
8,noisy_or_weak (шумный или слабый),spread_mean_reversion,5,1,0.522959,0.069492,-0.086447,93.475715,-0.039529,0.516919
12,trend_down (тренд вниз),inverse_spread_momentum,5,1,-0.489276,0.109903,-0.145341,-462.459316,-0.027433,0.228416
